In [2]:
#!/usr/bin/env python3
# =========================================================
# ENSEMBLE LEGAL SUMMARIZATION v4 — REAL KNOWLEDGE GRAPH
# =========================================================
# Architecture:
#   InLegalBERT + HSLN (RRC) → Real KG Construction (NetworkX)
#   → Graph Centrality Guided Sentence Selection
#   → KG-Attention (centrality-weighted logit boost) on LED
#   → KG-Diff Iterative Refinement (graph coverage criterion)
#   → LCS-Guided Sentence Fusion
#   → Verbatim Span Injection
#   → Structure-Aware Selective Paraphrasing
#   → Canonical Rhetorical Reconstruction
#
# Novel Contributions:
#   Idea 5+7 : KG-Diff + Semantic KG Coverage
#   Idea 8   : LCS-Guided Sentence Fusion
#   Idea 9   : Verbatim Span Injection
#   Idea 10  : KG-Attention During Generation
#   Idea 11  : Structure-Aware Selective Paraphrasing
#   ★ Idea 12: Real Per-Document Legal Knowledge Graph
#              - NetworkX DiGraph with typed nodes + edges
#              - PageRank / Betweenness / Degree centrality
#              - Graph Coverage-Based Sentence Selection
#              - Centrality-Proportional Entity Attention
#              - Graph-Coverage KG-Diff Refinement Loop
#              - Factual Consistency via Edge-Direction Check
#
# Dependencies:
#   pip install transformers torch torchcrf scikit-learn
#               pandas matplotlib seaborn tqdm evaluate
#               bert-score networkx spacy
#   python -m spacy download en_core_web_sm
# =========================================================

import os, json, re, math, time, random
from datetime import datetime
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import networkx as nx
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm
from nltk.tokenize import sent_tokenize, word_tokenize
from sklearn.metrics.pairwise import cosine_similarity
from transformers import (
    AutoTokenizer, AutoModel,
    AutoModelForSeq2SeqLM,
    LEDForConditionalGeneration,
    PegasusForConditionalGeneration,
    LogitsProcessor, LogitsProcessorList,
)
import evaluate
import nltk

# ── NLTK ──────────────────────────────────────────────────────────────────
for res in ['tokenizers/punkt', 'tokenizers/punkt_tab']:
    try:
        nltk.data.find(res)
    except LookupError:
        nltk.download(res.split('/')[-1], quiet=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🚀  Device: {device}")

# =========================================================
# ★ PATHS & GLOBAL CONFIG
# =========================================================
MODEL_PATHS = {
    "BART":       "BART",
    "LED":        "LED",
    "PEGASUS":    "PEGASUS",
    "ROLE_MODEL": "best_RR_model",
}
TRAIN_JSON_PATH = "train.json"
TEST_JSON_PATH  = "test.json"
OUTPUT_DIR      = "./ensemble_results_real_kg_v4"
os.makedirs(OUTPUT_DIR, exist_ok=True)

ROLE_CLASSIFICATION_FILE = "sentence_role_annotations_8label.json"
ROLE_CONTRIBUTION_FILE   = "role_contribution_scores_8label.json"
ROLE_WEIGHTS_FILE        = "normalized_role_weights_8label.json"
MAX_TRAIN_SAMPLES        = 3000

MODEL_CONFIG = {
    "BART":    {"max_input": 1024, "max_output": 256, "num_beams": 4},
    "LED":     {"max_input": 4096, "max_output": 512, "num_beams": 4},
    "PEGASUS": {"max_input": 1024, "max_output": 256, "num_beams": 4},
}

ENSEMBLE_WEIGHTS = {"BART": 0.25, "LED": 0.50, "PEGASUS": 0.25}

# ── 13-label → 8-label mapping ────────────────────────────────────────────
HSLN_LABELS = [
    "PREAMBLE","FAC","RLC","ISSUE","ARG_PETITIONER","ARG_RESPONDENT",
    "ANALYSIS","STA","PRE_RELIED","PRE_NOT_RELIED","RATIO","RPC","NONE",
]
NUM_HSLN_LABELS = 13

LABELS_8 = [
    "PREAMBLE","FACTS","ISSUE","ARGUMENTS",
    "REASONING","PRECEDENT_RELIED","PRECEDENT_NOT_RELIED","PROCEDURAL",
]
NUM_LABELS = 8

LABEL_MAPPING_13_TO_8 = {
    "PREAMBLE":"PREAMBLE","ISSUE":"ISSUE","FAC":"FACTS",
    "PRE_RELIED":"PRECEDENT_RELIED","PRE_NOT_RELIED":"PRECEDENT_NOT_RELIED",
    "ARG_PETITIONER":"ARGUMENTS","ARG_RESPONDENT":"ARGUMENTS",
    "ANALYSIS":"REASONING","RATIO":"REASONING","RPC":"REASONING",
    "STA":"PROCEDURAL","RLC":"PROCEDURAL","NONE":"PROCEDURAL",
}

def map_13_to_8(labels_13):
    return [LABEL_MAPPING_13_TO_8.get(l, "PROCEDURAL") for l in labels_13]

# ── Hybrid scoring weights ─────────────────────────────────────────────────
HYBRID_ALPHA = 0.5   # role weight
HYBRID_BETA  = 0.2   # centroid salience
HYBRID_GAMMA = 0.3   # ★ graph centrality (new)

# ── Compression ratios ────────────────────────────────────────────────────
COMPRESSION_RATIOS = {"BART":0.12,"PEGASUS":0.12,"LED":0.40}
MIN_SENTENCES      = {"BART":30,"PEGASUS":30,"LED":100}
MAX_SENTENCES      = {"BART":200,"PEGASUS":200,"LED":500}

def get_adaptive_targets(doc_len):
    return {
        m: max(MIN_SENTENCES[m], min(MAX_SENTENCES[m], int(doc_len * r)))
        for m, r in COMPRESSION_RATIOS.items()
    }

# =========================================================
# ★ IDEA 12 — REAL KNOWLEDGE GRAPH CONFIGURATION
# =========================================================

# Node type taxonomy
KG_NODE_TYPES = {
    "ENTITY":   "legal entity (party, court, authority)",
    "STATUTE":  "statutory provision (section, article, act)",
    "CITATION": "case citation (AIR, SCC, SCR)",
    "CONCEPT":  "legal concept (res judicata, mandamus)",
    "FACT":     "factual entity (date, amount, place)",
}

# Edge (relation) type taxonomy
KG_EDGE_TYPES = {
    "RELIED_ON":       "court relied on this precedent/provision",
    "SET_ASIDE":       "judgment/order was set aside",
    "CONVICTED_UNDER": "accused convicted under provision",
    "APPLIED_TO":      "provision applied to subject",
    "APPEALED_AGAINST":"appeal filed against decision",
    "GRANTED":         "relief/remedy granted",
    "REJECTED":        "relief/claim rejected",
    "INTERPRETED":     "provision interpreted as",
    "RELATED_TO":      "generic fallback relation",
}

# Regex patterns for typed node classification
_STATUTE_RE = re.compile(
    r'\b(?:Section|Article|Order|Rule|Schedule|Clause|Sub-section)'
    r'\s+\d+[\w\-\.]*(?:\s+(?:of\s+the\s+)?[A-Z][A-Za-z\s]+'
    r'(?:Act|Code|Rules|Regulations|Constitution|Penal Code))?\b',
    re.IGNORECASE,
)
_CITATION_RE = re.compile(
    r'\b(?:AIR|SCC|SCR|MLJ|Bom|Cal|Mad|Del|All|Ker|SC|HC)\s+\d{4}\s+\w+\s+\d+|'
    r'\(\d{4}\)\s+\d+\s+SCC\s+\d+|\d{4}\s+\(\d+\)\s+SCC\s+\d+',
    re.IGNORECASE,
)
_CONCEPT_RE = re.compile(
    r'\b(?:res judicata|sub judice|locus standi|ultra vires|prima facie|'
    r'inter alia|suo motu|ex parte|ad interim|mandamus|certiorari|'
    r'habeas corpus|prohibition|quo warranto|injunction|contempt)\b',
    re.IGNORECASE,
)
_FACT_RE = re.compile(
    r'\b(?:Rs\.?|₹|INR)\s*[\d,]+(?:\.\d+)?(?:\s*(?:lakhs?|crores?))?\b|'
    r'\b\d{1,2}[\/\-\.]\d{1,2}[\/\-\.]\d{2,4}\b',
    re.IGNORECASE,
)

# Performative verb → edge-type mapping
_VERB_EDGE_MAP = {
    "set aside":       "SET_ASIDE",
    "quash":           "SET_ASIDE",
    "quashed":         "SET_ASIDE",
    "rely":            "RELIED_ON",
    "relied":          "RELIED_ON",
    "convict":         "CONVICTED_UNDER",
    "convicted":       "CONVICTED_UNDER",
    "apply":           "APPLIED_TO",
    "applied":         "APPLIED_TO",
    "appeal":          "APPEALED_AGAINST",
    "appealed":        "APPEALED_AGAINST",
    "grant":           "GRANTED",
    "granted":         "GRANTED",
    "allow":           "GRANTED",
    "allowed":         "GRANTED",
    "dismiss":         "REJECTED",
    "dismissed":       "REJECTED",
    "reject":          "REJECTED",
    "rejected":        "REJECTED",
    "interpret":       "INTERPRETED",
    "interpreted":     "INTERPRETED",
}

# Graph centrality weights for hybrid score
KG_CENTRALITY_WEIGHTS = {
    "pagerank":    0.50,
    "betweenness": 0.30,
    "degree":      0.20,
}

# Minimum PageRank score for a node to be considered "important"
KG_IMPORTANT_NODE_THRESHOLD = 0.01

# Graph Coverage thresholds
KG_NODE_COVERAGE_THRESHOLD  = 0.80   # 80% of high-centrality nodes must be covered
KG_EDGE_COVERAGE_THRESHOLD  = 0.70   # 70% of important edges must be covered
KG_MAX_ITERATIONS            = 3
KG_TOP_MISSING               = 5
KG_MAX_CORRECTION_SENTS      = 10
KG_CRITICAL_ROLES            = ["ISSUE", "REASONING", "PRECEDENT_RELIED"]
KG_MIN_EDGE_WEIGHT           = 1     # prune edges with weight < this

# ── Sentence Fusion config ─────────────────────────────────────────────────
LCS_ANCHOR_ROLES          = ["ISSUE","REASONING","PRECEDENT_RELIED","FACTS"]
LCS_MIN_NGRAM_OVERLAP     = 3
LCS_MAX_OUTPUT_SENTENCES  = 20
LCS_ANCHOR_LCS_WEIGHT     = 0.5
LCS_ANCHOR_SEM_WEIGHT     = 0.5
LCS_CONNECTIVES = [
    "The court held that","It was observed that","Therefore,",
    "Furthermore,","The appellant contended that",
    "In view of the above,","The High Court noted that","Accordingly,",
]

# ── Verbatim span injection config ────────────────────────────────────────
VERBATIM_MIN_SPAN          = 4
VERBATIM_MAX_CONTEXT       = 10
VERBATIM_TARGET_ROLES      = ["ISSUE","REASONING","PRECEDENT_RELIED"]
VERBATIM_MAX_LENGTH_RATIO  = 2.0
VERBATIM_LOG_LIMIT         = 5

# ── KG-Attention (centrality-proportional) config ─────────────────────────
ENTITY_BASE_BOOST             = 2.0
ENTITY_MULTI_TOKEN_MULTIPLIER = 1.3
ENTITY_DECAY_FACTOR           = 0.5
ENTITY_MAX_BOOST              = 3.5   # slightly higher ceiling for centrality boost
ENTITY_MIN_TOKEN_LEN          = 2
ENTITY_BOOST_MODELS           = ["LED"]

# ── Structure-Aware Paraphrase config (Idea 11) ───────────────────────────
CANONICAL_ORDER = [
    "PREAMBLE","ISSUE","FACTS","ARGUMENTS",
    "REASONING","PRECEDENT_RELIED","PRECEDENT_NOT_RELIED","PROCEDURAL",
]
CANONICAL_POSITION       = {r: i for i, r in enumerate(CANONICAL_ORDER)}
ROLE_PARAPHRASE_AGGR     = {
    "PREAMBLE":            0.8,
    "FACTS":               None,  # skip
    "ISSUE":               0.1,
    "ARGUMENTS":           0.5,
    "REASONING":           0.4,
    "PRECEDENT_RELIED":    0.2,
    "PRECEDENT_NOT_RELIED":0.0,
    "PROCEDURAL":          0.6,
}
LCS_REGRESSION_TOLERANCE = 0.02
LEGAL_ANCHOR_THRESHOLD   = 0.6
MIN_AGGR_TO_PARAPHRASE   = 0.15
PARAPHRASE_INSTRUCTION   = (
    "Paraphrase only the connecting words and transition phrases. "
    "Keep all legal citations, party names, section numbers, and "
    "judicial terms exactly as they are: "
)

# ── Legal anchor seed tokens for prototype embedding ─────────────────────
LEGAL_ANCHOR_SEED_TOKENS = [
    "Section","Article","Act","Code","SCC","AIR","quashed",
    "dismissed","allowed","upheld","petitioner","respondent",
    "appellant","writ","mandamus","certiorari","injunction",
    "Constitution","Supreme","jurisdiction","decree","warrant",
]

LEGAL_ANCHOR_PATTERNS = [
    re.compile(
        r'\b(Section|Sections|Article|Articles|Order|Rule|Rules|Schedule|'
        r'Clause|Sub-section|Sub-clause|Para|Chapter)\s+\d+[\w\-\.]*'
        r'(?:\s+(?:of\s+the\s+)?[A-Z][A-Za-z\s]+(?:Act|Code|Rules|'
        r'Regulations|Ordinance|Constitution|Penal Code))?',
        re.IGNORECASE,
    ),
    re.compile(
        r'\b(?:the\s+)?[A-Z][A-Za-z\s]+(?:Act|Code|Rules|Regulations|'
        r'Ordinance|Constitution|Penal Code|Procedure Code|Evidence Act)\b',
        re.IGNORECASE,
    ),
    re.compile(
        r'\b(?:AIR|SCC|SCR|MLJ|Bom|Cal|Mad|Del|All|Ker|SC|HC)'
        r'\s+\d{4}\s+\w+\s+\d+',re.IGNORECASE,
    ),
    re.compile(r'\(\d{4}\)\s+\d+\s+SCC\s+\d+',re.IGNORECASE),
    re.compile(r'(?:Rs\.?|₹|INR|USD|EUR)\s*[\d,]+(?:\.\d+)?'
               r'\s*(?:lakhs?|crores?|thousands?|millions?)?',re.IGNORECASE),
    re.compile(r'\b\d+(?:,\d+)*(?:\.\d+)?\s*(?:lakhs?|crores?)\b',re.IGNORECASE),
    re.compile(
        r'\b(?:quashed|set aside|restored|remanded|dismissed|allowed|'
        r'upheld|confirmed|reversed|modified|stayed|vacated|granted|'
        r'rejected|affirmed|overruled)\b',re.IGNORECASE,
    ),
    re.compile(
        r'\b(?:Supreme Court|High Court|District Court|Sessions Court|'
        r'Civil Court|Criminal Court|Tribunal|Commission|Authority|Board)\b'
        r'(?:\s+of\s+[A-Z][A-Za-z]+)?',re.IGNORECASE,
    ),
    re.compile(
        r'\b\d{1,2}(?:st|nd|rd|th)?\s+(?:January|February|March|April|May|'
        r'June|July|August|September|October|November|December)\s+\d{4}\b',
        re.IGNORECASE,
    ),
    re.compile(r'\b\d{1,2}[\/\-\.]\d{1,2}[\/\-\.]\d{2,4}\b'),
    re.compile(
        r'\b(?:writ of\s+)?(?:mandamus|certiorari|habeas corpus|prohibition|'
        r'quo warranto|injunction|attachment|execution|contempt)\b',
        re.IGNORECASE,
    ),
    re.compile(
        r'\b(?:res judicata|sub judice|locus standi|ultra vires|prima facie|'
        r'inter alia|inter se|suo motu|ex parte|ad interim|ad hoc)\b',
        re.IGNORECASE,
    ),
]

CEREMONY_PATTERNS = [
    re.compile(r'^(?:It is (?:pertinent|relevant|noteworthy|significant|important)'
               r' to (?:note|mention|observe|state) that\s*)',re.IGNORECASE),
    re.compile(r'^(?:Having (?:regard|considered|heard) (?:to|the) (?:the\s+)?'
               r'(?:above|foregoing|submissions?|arguments?)[,\s]+)',re.IGNORECASE),
    re.compile(r'^(?:In (?:the|this) (?:context|regard|view|light|background|'
               r'present case)[,\s]+)',re.IGNORECASE),
    re.compile(r'^(?:(?:After|Upon|On) (?:careful|due|detailed|thorough) '
               r'(?:consideration|examination|perusal|deliberation)[,\s]+)',re.IGNORECASE),
    re.compile(r'^(?:For (?:all|the) (?:the\s+)?(?:aforesaid|aforementioned|'
               r'foregoing|above) (?:reasons?|grounds?)[,\s]+)',re.IGNORECASE),
    re.compile(r'^(?:The (?:Hon\'?ble|learned) (?:Court|Judge|Bench|Division Bench|'
               r'Full Bench)\s+(?:was pleased to\s+)?)',re.IGNORECASE),
]

_LEGAL_ANCHOR_PROTOTYPE = None

# =========================================================
# HSLN MODEL DEFINITION
# =========================================================

class PrototypeAttention(nn.Module):
    def __init__(self, dim, heads=8, dropout=0.1):
        super().__init__()
        self.h  = heads
        self.dh = dim // heads
        self.q  = nn.Linear(dim, dim, bias=False)
        self.k  = nn.Linear(dim, dim, bias=False)
        self.v  = nn.Linear(dim, dim, bias=False)
        self.o  = nn.Linear(dim, dim, bias=False)
        self.ln = nn.LayerNorm(dim)
        self.dp = nn.Dropout(dropout)

    def forward(self, x, p):
        B, T, D = x.shape
        C = p.size(0)
        Q = self.q(x).view(B, T, self.h, self.dh)
        K = self.k(p).view(C, self.h, self.dh)
        V = self.v(p).view(C, self.h, self.dh)
        Q = F.normalize(Q, -1); K = F.normalize(K, -1)
        outs = []
        for h in range(self.h):
            a = (Q[:,:,h] @ K[:,h].T).softmax(-1)
            outs.append(self.dp(a) @ V[:,h])
        return self.ln(x + self.dp(self.o(torch.cat(outs,-1))))

class RPL(nn.Module):
    def __init__(self, dim, num_labels):
        super().__init__()
        self.proto = nn.Parameter(torch.randn(num_labels, dim))
    def forward(self, h):
        return F.normalize(h,-1) @ F.normalize(self.proto,-1).T

class RTM(nn.Module):
    def __init__(self, num_labels, lam):
        super().__init__()
        self.A   = nn.Parameter(torch.zeros(num_labels, num_labels))
        self.lam = lam
    def forward(self, logits):
        lp = logits.log_softmax(-1)
        for t in range(1, logits.size(1)):
            tr = torch.logsumexp(lp[:,t-1].unsqueeze(1) + self.A.log_softmax(-1),-1)
            logits[:,t] += self.lam * tr
        return logits

class HSLNModel(nn.Module):
    def __init__(self, embedding_dim=768, lstm_hidden=384, num_labels=13,
                 dropout=0.3, rtm_lambda=0.05):
        super().__init__()
        self.att  = PrototypeAttention(embedding_dim, dropout=dropout)
        self.lstm = nn.LSTM(embedding_dim, lstm_hidden, 2, bidirectional=True,
                            batch_first=True, dropout=dropout)
        self.cls  = nn.Linear(lstm_hidden*2, num_labels)
        self.rpl  = RPL(lstm_hidden*2, num_labels)
        self.alpha= nn.Parameter(torch.tensor(2.0))
        self.rtm  = RTM(num_labels, rtm_lambda)
        self.register_buffer('prototypes', torch.randn(num_labels, embedding_dim))

    def forward(self, embeddings, lengths=None):
        x = self.att(embeddings, self.prototypes)
        if lengths is not None:
            pck  = nn.utils.rnn.pack_padded_sequence(x, lengths.cpu(),
                                                      batch_first=True, enforce_sorted=False)
            o, _ = self.lstm(pck)
            o, _ = nn.utils.rnn.pad_packed_sequence(o, batch_first=True)
        else:
            o, _ = self.lstm(x)
        a = torch.sigmoid(self.alpha)
        return self.rtm(a*self.cls(o) + (1-a)*self.rpl(o))

    def predict(self, embeddings, lengths=None):
        return torch.argmax(self.forward(embeddings, lengths), dim=-1)

# =========================================================
# LOAD HSLN ROLE CLASSIFIER
# =========================================================
print("\n📚 Loading HSLN role classifier...")
cfg = {}
cp  = os.path.join(MODEL_PATHS["ROLE_MODEL"], "config.json")
if os.path.exists(cp):
    with open(cp) as f: cfg = json.load(f)

role_model = HSLNModel(
    embedding_dim=cfg.get('embedding_dim',768),
    lstm_hidden  =cfg.get('lstm_hidden',384),
    num_labels   =NUM_HSLN_LABELS,
    dropout      =cfg.get('dropout',0.3),
    rtm_lambda   =cfg.get('rtm_lambda',0.05),
)
sd = torch.load(os.path.join(MODEL_PATHS["ROLE_MODEL"],"pytorch_model.bin"), map_location=device)
sd.pop('prototypes', None)
role_model.load_state_dict(sd, strict=False)
role_model.prototypes = torch.load(
    os.path.join(MODEL_PATHS["ROLE_MODEL"],"prototypes.pt"), map_location=device)
role_model.to(device).eval()
print("✅  HSLN role classifier loaded")

# =========================================================
# LOAD InLegalBERT
# =========================================================
print("\n📚 Loading InLegalBERT...")
legal_tokenizer = AutoTokenizer.from_pretrained("law-ai/InLegalBERT")
legal_model     = AutoModel.from_pretrained("law-ai/InLegalBERT").to(device)
legal_model.eval()
print("✅  InLegalBERT loaded")

# =========================================================
# EMBEDDING UTILS
# =========================================================
@torch.no_grad()
def embed_with_legalbert(texts, batch_size=16):
    if not texts: return np.zeros((0, 768))
    embs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        enc   = legal_tokenizer(batch, padding=True, truncation=True,
                                max_length=512, return_tensors="pt").to(device)
        out   = legal_model(**enc).last_hidden_state
        mask  = enc["attention_mask"].unsqueeze(-1).float()
        embs.append(((out*mask).sum(1)/mask.sum(1)).cpu().numpy())
    return np.vstack(embs)

@torch.no_grad()
def classify_roles(sentences, batch_size=16):
    if not sentences: return []
    preds = []
    for i in range(0, len(sentences), batch_size):
        batch = sentences[i:i+batch_size]
        inp   = legal_tokenizer(batch, padding=True, truncation=True,
                                max_length=128, return_tensors="pt").to(device)
        emb   = legal_model(**inp).last_hidden_state.mean(1).unsqueeze(1)
        lens  = torch.ones(len(batch), dtype=torch.long).to(device)
        preds.extend(role_model.predict(emb, lens)[:,0].cpu().tolist())
    return map_13_to_8([HSLN_LABELS[p] for p in preds])

# =========================================================
# ★ IDEA 12 — REAL KNOWLEDGE GRAPH CONSTRUCTION
# =========================================================

def _classify_node_type(entity_str):
    """Classify an entity string into KG node types."""
    s = entity_str.strip()
    if _STATUTE_RE.search(s):  return "STATUTE"
    if _CITATION_RE.search(s): return "CITATION"
    if _CONCEPT_RE.search(s):  return "CONCEPT"
    if _FACT_RE.search(s):     return "FACT"
    return "ENTITY"

def _get_edge_type_from_verb(verb_str):
    """Map a lemmatised verb to a typed edge relation."""
    v = verb_str.lower().strip()
    for phrase, etype in _VERB_EDGE_MAP.items():
        if phrase in v:
            return etype
    return "RELATED_TO"

def extract_typed_triples(sentences, use_spacy=True):
    """
    Extract (subject, relation_type, object, source_sent_idx) typed triples.

    Two modes:
      1. spaCy dependency parse (precise, typed)
      2. Regex + noun co-occurrence fallback

    Returns list of dicts:
      { 'subj', 'rel', 'obj', 'edge_type', 'sent_idx', 'sentence' }
    """
    triples = []

    # ── Mode 1: spaCy ─────────────────────────────────────────────────────
    if use_spacy:
        try:
            import spacy
            try:    nlp = spacy.load("en_legal_ner_trf")
            except: nlp = spacy.load("en_core_web_sm")

            for sent_idx, sent in enumerate(sentences):
                if not sent.strip(): continue
                doc = nlp(sent[:512])
                for token in doc:
                    if token.dep_ == "ROOT" and token.pos_ == "VERB":
                        subjs = [c for c in token.children
                                 if c.dep_ in ("nsubj","nsubjpass")]
                        objs  = [c for c in token.children
                                 if c.dep_ in ("dobj","pobj","attr","oprd")]
                        for s in subjs:
                            for o in objs:
                                subj_text = " ".join(
                                    t.text for t in s.subtree if not t.is_stop
                                ).lower().strip()
                                obj_text  = " ".join(
                                    t.text for t in o.subtree if not t.is_stop
                                ).lower().strip()
                                rel_verb  = token.lemma_.lower()
                                edge_type = _get_edge_type_from_verb(rel_verb)
                                if subj_text and obj_text:
                                    triples.append({
                                        "subj":      subj_text,
                                        "rel":       rel_verb,
                                        "obj":       obj_text,
                                        "edge_type": edge_type,
                                        "sent_idx":  sent_idx,
                                        "sentence":  sent,
                                    })
            return triples
        except Exception:
            pass

    # ── Mode 2: Regex fallback ────────────────────────────────────────────
    for sent_idx, sent in enumerate(sentences):
        words = re.findall(r'\b[A-Za-z][a-z]+(?:\s+[A-Z][a-z]+)*\b', sent)
        words = [w.lower() for w in words if len(w) > 3]
        for i in range(len(words)-1):
            triples.append({
                "subj":      words[i],
                "rel":       "related_to",
                "obj":       words[i+1],
                "edge_type": "RELATED_TO",
                "sent_idx":  sent_idx,
                "sentence":  sent,
            })
    return triples


def build_legal_kg(doc_annotation, roles_to_include=None):
    """
    ★ IDEA 12 — Build a real per-document Legal Knowledge Graph.

    Constructs a NetworkX DiGraph where:
      - Nodes  = legal entities with type, role, frequency attributes
      - Edges  = typed legal relations with weight (co-occurrence count)
                 and provenance (sentence index)

    Args:
        doc_annotation:   dict  — sentence-role annotations
        roles_to_include: list  — roles to include in KG (None = all)

    Returns:
        G:              nx.DiGraph  — the knowledge graph
        node_meta:      dict        — node → {type, role, freq, sentences}
        edge_meta:      dict        — (u,v) → {edge_type, weight, sent_indices}
        build_log:      dict        — diagnostic info
    """
    build_log = {
        "method":             "real_kg_construction (Idea 12)",
        "total_source_sents": len(doc_annotation["sentences"]),
        "roles_included":     roles_to_include or "ALL",
    }

    # Filter sentences by role
    if roles_to_include:
        source_sents = [
            (s["index"], s["sentence"], s["role"])
            for s in doc_annotation["sentences"]
            if s["role"] in roles_to_include
        ]
    else:
        source_sents = [
            (s["index"], s["sentence"], s["role"])
            for s in doc_annotation["sentences"]
        ]

    build_log["filtered_sents"] = len(source_sents)

    if not source_sents:
        G = nx.DiGraph()
        return G, {}, {}, build_log

    sent_texts = [s[1] for s in source_sents]
    sent_roles = {s[0]: s[2] for s in source_sents}

    # Extract typed triples
    raw_triples = extract_typed_triples(sent_texts, use_spacy=True)
    build_log["raw_triples_extracted"] = len(raw_triples)

    # ── Build graph ────────────────────────────────────────────────────────
    G         = nx.DiGraph()
    node_meta = {}  # node_id → {type, role, freq, sentences}
    edge_meta = {}  # (u,v)   → {edge_type, weight, sent_indices}

    for triple in raw_triples:
        subj      = triple["subj"].strip().lower()
        obj       = triple["obj"].strip().lower()
        edge_type = triple["edge_type"]
        sent_idx  = triple["sent_idx"]
        role      = sent_roles.get(sent_idx, "REASONING")

        if not subj or not obj or len(subj) < 2 or len(obj) < 2:
            continue

        # ── Add / update nodes ─────────────────────────────────────────────
        for node_str, is_subj in [(subj, True), (obj, False)]:
            if node_str not in node_meta:
                node_meta[node_str] = {
                    "type":      _classify_node_type(node_str),
                    "role":      role,
                    "freq":      0,
                    "sentences": [],
                }
            node_meta[node_str]["freq"]      += 1
            node_meta[node_str]["sentences"].append(sent_idx)
            if node_str not in G:
                G.add_node(node_str,
                           node_type=node_meta[node_str]["type"],
                           role=role,
                           freq=1)
            else:
                G.nodes[node_str]["freq"] = node_meta[node_str]["freq"]

        # ── Add / update edge ──────────────────────────────────────────────
        edge_key = (subj, obj)
        if G.has_edge(subj, obj):
            G[subj][obj]["weight"]        += 1
            G[subj][obj]["sent_indices"].append(sent_idx)
            edge_meta[edge_key]["weight"] += 1
            edge_meta[edge_key]["sent_indices"].append(sent_idx)
        else:
            G.add_edge(subj, obj,
                       edge_type=edge_type,
                       relation=triple["rel"],
                       weight=1,
                       sent_indices=[sent_idx])
            edge_meta[edge_key] = {
                "edge_type":   edge_type,
                "relation":    triple["rel"],
                "weight":      1,
                "sent_indices":[sent_idx],
            }

    # ── Prune weak edges ───────────────────────────────────────────────────
    weak_edges = [(u,v) for u,v,d in G.edges(data=True)
                  if d["weight"] < KG_MIN_EDGE_WEIGHT]
    G.remove_edges_from(weak_edges)

    # Remove isolated nodes (after pruning)
    isolated = list(nx.isolates(G))
    G.remove_nodes_from(isolated)

    build_log["nodes_after_pruning"] = G.number_of_nodes()
    build_log["edges_after_pruning"] = G.number_of_edges()
    return G, node_meta, edge_meta, build_log


# =========================================================
# ★ IDEA 12 — GRAPH CENTRALITY COMPUTATION
# =========================================================

def compute_graph_centrality(G):
    """
    Compute three centrality measures and combine into a unified
    importance score per node.

    Returns:
        centrality_scores: dict  — node → unified_score (0–1 range)
        raw_centrality:    dict  — node → {pagerank, betweenness, degree}
    """
    if G.number_of_nodes() == 0:
        return {}, {}

    # ── PageRank ──────────────────────────────────────────────────────────
    try:
        pr = nx.pagerank(G, alpha=0.85, weight='weight', max_iter=200)
    except Exception:
        pr = {n: 1.0/G.number_of_nodes() for n in G.nodes()}

    # ── Betweenness centrality ────────────────────────────────────────────
    # Expensive on large graphs — use approximate k-sample version
    n_nodes = G.number_of_nodes()
    k_sample = min(n_nodes, max(10, n_nodes // 5))
    try:
        bc = nx.betweenness_centrality(G, k=k_sample, weight='weight',
                                        normalized=True, seed=42)
    except Exception:
        bc = {n: 0.0 for n in G.nodes()}

    # ── Degree centrality ─────────────────────────────────────────────────
    dc = nx.degree_centrality(G)

    # ── Normalise each measure to [0,1] ───────────────────────────────────
    def norm_dict(d):
        vals = list(d.values())
        mn, mx = min(vals), max(vals)
        if mx == mn: return {k: 0.5 for k in d}
        return {k: (v-mn)/(mx-mn) for k,v in d.items()}

    pr_n = norm_dict(pr)
    bc_n = norm_dict(bc)
    dc_n = norm_dict(dc)

    w = KG_CENTRALITY_WEIGHTS
    unified = {}
    raw     = {}
    for node in G.nodes():
        score = (w["pagerank"]    * pr_n.get(node, 0) +
                 w["betweenness"] * bc_n.get(node, 0) +
                 w["degree"]      * dc_n.get(node, 0))
        unified[node] = score
        raw[node] = {
            "pagerank":    round(pr.get(node,0), 6),
            "betweenness": round(bc.get(node,0), 6),
            "degree":      round(dc.get(node,0), 6),
            "unified":     round(score, 6),
        }

    return unified, raw


# =========================================================
# ★ IDEA 12 — GRAPH COVERAGE COMPUTATION
# =========================================================

def compute_graph_coverage(G, centrality_scores, summary_text,
                            anchor_prototype, node_threshold=KG_IMPORTANT_NODE_THRESHOLD):
    """
    Compute what fraction of important KG nodes and edges are
    semantically covered by the current summary.

    Node coverage: a node is covered if its string appears in the summary
                   OR a summary sentence embedding is close to its embedding.
    Edge coverage: an edge (u,v,rel) is covered if both endpoints covered
                   AND the relation is expressed (simple heuristic).

    Returns:
        node_coverage:    float  — fraction of important nodes covered
        edge_coverage:    float  — fraction of important edges covered
        uncovered_nodes:  list   — [(node, centrality)] sorted by importance
        uncovered_edges:  list   — [(u, v, edge_type, centrality_sum)]
        coverage_log:     dict
    """
    summary_lower   = summary_text.lower()
    summary_sents   = sent_tokenize(summary_text)

    # Important nodes: above threshold
    important_nodes = {
        n: s for n, s in centrality_scores.items()
        if s >= node_threshold and G.has_node(n)
    }

    if not important_nodes:
        return 1.0, 1.0, [], [], {"note": "No important nodes above threshold"}

    # ── Node coverage ──────────────────────────────────────────────────────
    covered_nodes   = set()
    uncovered_nodes = []

    for node, score in important_nodes.items():
        # Check 1: exact string match
        if node in summary_lower:
            covered_nodes.add(node)
            continue

        # Check 2: any word of multi-word node appears
        node_words = node.split()
        if len(node_words) > 1 and any(w in summary_lower
                                        for w in node_words if len(w) > 4):
            covered_nodes.add(node)
            continue

        uncovered_nodes.append((node, score))

    uncovered_nodes.sort(key=lambda x: x[1], reverse=True)
    node_cov = len(covered_nodes) / max(len(important_nodes), 1)

    # ── Edge coverage ──────────────────────────────────────────────────────
    important_edges  = [
        (u, v, d["edge_type"],
         important_nodes.get(u, 0) + important_nodes.get(v, 0))
        for u, v, d in G.edges(data=True)
        if u in important_nodes and v in important_nodes
    ]
    important_edges.sort(key=lambda x: x[3], reverse=True)

    covered_edges   = 0
    uncovered_edges = []

    for u, v, etype, csum in important_edges:
        if u in covered_nodes and v in covered_nodes:
            covered_edges += 1
        else:
            uncovered_edges.append((u, v, etype, csum))

    edge_cov = (covered_edges / max(len(important_edges), 1)
                if important_edges else 1.0)

    coverage_log = {
        "important_nodes":  len(important_nodes),
        "covered_nodes":    len(covered_nodes),
        "node_coverage":    round(node_cov, 4),
        "important_edges":  len(important_edges),
        "covered_edges":    covered_edges,
        "edge_coverage":    round(edge_cov, 4),
        "top_uncovered":    [(n, round(s,4)) for n,s in uncovered_nodes[:5]],
    }
    return node_cov, edge_cov, uncovered_nodes, uncovered_edges, coverage_log


# =========================================================
# ★ IDEA 12 — GRAPH CENTRALITY GUIDED SENTENCE SELECTION
# (Replaces pure hybrid role-salience selection for the KG path)
# =========================================================

def graph_centrality_guided_selection(doc_annotation, normalized_weights,
                                       G, centrality_scores, target_sentences):
    """
    ★ Enhanced sentence selection combining:
      - Role weight (HYBRID_ALPHA)
      - Centroid cosine salience (HYBRID_BETA)
      - ★ Graph centrality of entities in sentence (HYBRID_GAMMA)
      - ★ Graph coverage gain (greedy set-cover bonus)

    Args:
        doc_annotation:    dict  — sentence-role annotations
        normalized_weights:dict  — role → weight
        G:                 DiGraph — per-document KG
        centrality_scores: dict  — node → unified centrality
        target_sentences:  int   — how many to select

    Returns:
        filtered_text: str
        selection_info: dict
    """
    sentences = [s["sentence"] for s in doc_annotation["sentences"]]
    roles     = [s["role"]     for s in doc_annotation["sentences"]]

    if not sentences:
        return "", {"selected": 0}

    embeddings   = embed_with_legalbert(sentences)
    centroid     = embeddings.mean(axis=0, keepdims=True)
    similarities = cosine_similarity(embeddings, centroid).squeeze()
    if similarities.ndim == 0:
        similarities = np.array([float(similarities)])

    # ── Score every sentence ───────────────────────────────────────────────
    covered_nodes = set()  # tracks greedy coverage
    scored        = []

    for idx, (sent, role, sim) in enumerate(zip(sentences, roles, similarities)):
        role_w  = normalized_weights.get(role, 0)
        sent_lc = sent.lower()

        # Graph centrality of entities mentioned in this sentence
        sent_centralities = [
            score for node, score in centrality_scores.items()
            if node in sent_lc
        ]
        avg_centrality = np.mean(sent_centralities) if sent_centralities else 0.0

        # Coverage gain: how many NEW important nodes does this sentence add?
        new_nodes = sum(
            1 for node in centrality_scores
            if node in sent_lc and node not in covered_nodes
              and centrality_scores[node] >= KG_IMPORTANT_NODE_THRESHOLD
        )
        coverage_gain = new_nodes / max(len(centrality_scores), 1)

        hybrid_score = (
            HYBRID_ALPHA * role_w * 10 +
            HYBRID_BETA  * float(sim) +
            HYBRID_GAMMA * (avg_centrality + coverage_gain)
        )
        scored.append({
            "index":          idx,
            "sentence":       sent,
            "role":           role,
            "hybrid_score":   hybrid_score,
            "avg_centrality": avg_centrality,
            "coverage_gain":  coverage_gain,
        })

    # ── Greedy selection ───────────────────────────────────────────────────
    sorted_sents     = sorted(scored, key=lambda x: x["hybrid_score"], reverse=True)
    selected_indices = sorted([s["index"] for s in sorted_sents[:target_sentences]])
    selected_sents   = [sentences[i] for i in selected_indices]

    # Update coverage tracker
    for idx in selected_indices:
        sent_lc = sentences[idx].lower()
        for node in centrality_scores:
            if node in sent_lc:
                covered_nodes.add(node)

    selection_info = {
        "method":         "graph_centrality_guided",
        "selected":       len(selected_indices),
        "covered_nodes":  len(covered_nodes),
        "total_nodes":    len(centrality_scores),
    }
    return " ".join(selected_sents), selection_info


def hybrid_role_salience_selection(doc_annotation, normalized_weights,
                                    target_sentences):
    """Standard hybrid selection (no KG) — used for non-KG baselines."""
    sentences    = [s["sentence"] for s in doc_annotation["sentences"]]
    roles        = [s["role"]     for s in doc_annotation["sentences"]]
    if not sentences: return "", {}
    embs     = embed_with_legalbert(sentences)
    centroid = embs.mean(0, keepdims=True)
    sims     = cosine_similarity(embs, centroid).squeeze()
    if sims.ndim == 0: sims = np.array([float(sims)])
    scored   = [
        {"index": i, "score": HYBRID_ALPHA*normalized_weights.get(r,0)*10
                               + HYBRID_BETA*float(s), "sentence": sentences[i]}
        for i, (r, s) in enumerate(zip(roles, sims))
    ]
    sel_idx  = sorted([x["index"] for x in
                        sorted(scored, key=lambda x: x["score"], reverse=True)
                        [:target_sentences]])
    return " ".join(sentences[i] for i in sel_idx), {"selected": len(sel_idx)}



# =========================================================
# ★ IDEA 12 — CENTRALITY-PROPORTIONAL KG-ATTENTION
# =========================================================

def extract_entities_with_centrality(G, centrality_scores, tokenizer):
    """
    Extract entity strings from the KG, weighted by their centrality score.
    Returns list of (entity_string, centrality_score) sorted by score desc.
    Only includes nodes above the importance threshold.
    """
    entity_list = []
    for node, score in centrality_scores.items():
        if score >= KG_IMPORTANT_NODE_THRESHOLD and G.has_node(node):
            entity_list.append((node, score))
    entity_list.sort(key=lambda x: x[1], reverse=True)
    return entity_list


class CentralityKGEntityBoost(LogitsProcessor):
    """
    ★ IDEA 12 — Centrality-Proportional KG-Attention LogitsProcessor.

    Unlike the flat boost in Idea 10, this version scales each entity's
    boost by its normalised graph centrality score:

        base_boost(entity) = ENTITY_BASE_BOOST × centrality_norm(entity)
                           × (MULTI_TOKEN_MULT if multi-token)

    High-centrality nodes (e.g. "Section 302 IPC" with PageRank 0.8)
    get a strong boost. Low-centrality nodes (e.g. "learned counsel"
    with PageRank 0.05) get a weak boost.

    Boost decays per occurrence in prefix (ENTITY_DECAY_FACTOR^count)
    and is clamped to ENTITY_MAX_BOOST.
    """

    def __init__(self, entity_list, tokenizer,
                 base_boost=ENTITY_BASE_BOOST,
                 multi_mult=ENTITY_MULTI_TOKEN_MULTIPLIER,
                 decay_factor=ENTITY_DECAY_FACTOR,
                 max_boost=ENTITY_MAX_BOOST):
        self.decay_factor = decay_factor
        self.max_boost    = max_boost
        self.token_boost  = {}

        if not entity_list:
            self.boosted_ids    = []
            self.boosted_boosts = []
            return

        # Normalise centrality scores to [0,1] across the entity list
        scores = [s for _, s in entity_list]
        s_min, s_max = min(scores), max(scores)
        s_range      = s_max - s_min if s_max > s_min else 1.0

        for entity_str, raw_score in entity_list:
            centrality_norm  = (raw_score - s_min) / s_range
            token_ids        = tokenizer.encode(entity_str, add_special_tokens=False)
            is_multi         = len(token_ids) > 1
            # Centrality-proportional boost
            effective_boost  = base_boost * centrality_norm
            if is_multi:
                effective_boost *= multi_mult
            effective_boost = max(effective_boost, base_boost * 0.1)  # floor

            for tid in token_ids:
                self.token_boost[tid] = max(
                    self.token_boost.get(tid, 0.0), effective_boost
                )

        self.boosted_ids    = list(self.token_boost.keys())
        self.boosted_boosts = [self.token_boost[tid] for tid in self.boosted_ids]

    def __call__(self, input_ids, scores):
        if not self.boosted_ids:
            return scores
        for b in range(scores.shape[0]):
            prefix = input_ids[b]
            for i, tid in enumerate(self.boosted_ids):
                base  = self.boosted_boosts[i]
                count = (prefix == tid).sum().item()
                eff   = base * (self.decay_factor ** count)
                eff   = min(max(eff, 0.0), self.max_boost)
                scores[b, tid] += eff
        return scores


def generate_summary_kg_centrality_attention(model_name, filtered_text,
                                              G, centrality_scores):
    """
    ★ Generate summary with centrality-proportional KG-Attention.
    Applied only to LED (highest-quality baseline).
    Other models use standard generation.
    """
    model     = models[model_name]
    tokenizer = tokenizers[model_name]
    config    = MODEL_CONFIG[model_name]

    inputs = tokenizer(filtered_text, return_tensors="pt",
                       truncation=True, max_length=config["max_input"]).to(device)

    if model_name in ENTITY_BOOST_MODELS and G.number_of_nodes() > 0:
        entity_list      = extract_entities_with_centrality(G, centrality_scores, tokenizer)
        kg_processor     = CentralityKGEntityBoost(entity_list, tokenizer)
        logits_processor = LogitsProcessorList([kg_processor])
    else:
        logits_processor = LogitsProcessorList([])

    with torch.no_grad():
        if model_name == "LED":
            gam = torch.zeros_like(inputs["attention_mask"])
            gam[:, 0] = 1
            out_ids = model.generate(
                inputs["input_ids"],
                attention_mask=inputs["attention_mask"],
                global_attention_mask=gam,
                max_length=config["max_output"],
                num_beams=config["num_beams"],
                early_stopping=True,
                no_repeat_ngram_size=3,
                logits_processor=logits_processor,
            )
        else:
            out_ids = model.generate(
                inputs["input_ids"],
                attention_mask=inputs["attention_mask"],
                max_length=config["max_output"],
                num_beams=config["num_beams"],
                early_stopping=True,
                no_repeat_ngram_size=3,
                logits_processor=logits_processor,
            )
    return tokenizer.decode(out_ids[0], skip_special_tokens=True)


# =========================================================
# ★ IDEA 12 — GRAPH COVERAGE KG-DIFF REFINEMENT
# =========================================================

def kg_graph_coverage_refinement(summaries_dict, doc_annotation,
                                  G, centrality_scores,
                                  anchor_prototype,
                                  max_iterations=KG_MAX_ITERATIONS):
    """
    ★ IDEA 12 — KG-Diff with real graph coverage criterion.

    Improvement over pseudo-KG version:
      - Coverage check uses actual graph nodes (not just triple strings)
      - Refinement priority follows node centrality (highest uncovered first)
      - Edge-direction consistency check (basic factual validation)
      - Correction sentences chosen by: they contain uncovered high-centrality nodes

    Loop:
      1. Compute node + edge coverage of current summary
      2. If both thresholds met → stop
      3. Find top-k uncovered nodes (by centrality)
      4. Find source sentences containing those nodes (prioritised by role)
      5. Use PEGASUS to refine summary with correction context
      6. Accept if graph coverage improved
    """
    refine_log = {
        "method":     "graph_coverage_kg_diff (Idea 12)",
        "iterations": [],
        "kg_nodes":   G.number_of_nodes(),
        "kg_edges":   G.number_of_edges(),
    }

    if G.number_of_nodes() == 0:
        refine_log["early_exit"] = "Empty KG"
        return summaries_dict.get("LED_kg_attn",
               summaries_dict.get("LED", "")), refine_log

    current_summary = (summaries_dict.get("LED_kg_attn","") or
                       summaries_dict.get("LED","") or
                       summaries_dict.get("PEGASUS",""))

    for iteration in range(max_iterations):
        node_cov, edge_cov, uncov_nodes, uncov_edges, cov_log = \
            compute_graph_coverage(G, centrality_scores, current_summary,
                                   anchor_prototype)

        iter_log = {
            "iteration":    iteration + 1,
            "node_coverage": round(node_cov, 4),
            "edge_coverage": round(edge_cov, 4),
            "coverage_log":  cov_log,
        }

        # Check termination
        if (node_cov >= KG_NODE_COVERAGE_THRESHOLD and
                edge_cov >= KG_EDGE_COVERAGE_THRESHOLD):
            iter_log["action"] = (
                f"STOPPED — node_cov={node_cov:.3f}≥{KG_NODE_COVERAGE_THRESHOLD}, "
                f"edge_cov={edge_cov:.3f}≥{KG_EDGE_COVERAGE_THRESHOLD}"
            )
            refine_log["iterations"].append(iter_log)
            break

        if not uncov_nodes:
            iter_log["action"] = "STOPPED — all important nodes covered"
            refine_log["iterations"].append(iter_log)
            break

        # Top-k uncovered nodes by centrality
        target_nodes = [n for n, _ in uncov_nodes[:KG_TOP_MISSING]]
        iter_log["target_uncovered_nodes"] = target_nodes

        # Find source sentences covering target nodes (role-prioritised)
        correction_sents = []
        for s in doc_annotation["sentences"]:
            sent_lc = s["sentence"].lower()
            if any(node in sent_lc for node in target_nodes):
                priority = 0 if s["role"] in KG_CRITICAL_ROLES else 1
                correction_sents.append((priority, s["sentence"]))

        correction_sents.sort(key=lambda x: x[0])
        correction_sents = [s for _, s in correction_sents[:KG_MAX_CORRECTION_SENTS]]

        if not correction_sents:
            iter_log["action"] = "STOPPED — no correction sentences found"
            refine_log["iterations"].append(iter_log)
            break

        correction_input = (
            "Improve this legal summary by including the missing information.\n\n"
            f"Current summary: {current_summary}\n\n"
            f"Missing information: {' '.join(correction_sents)}\n\n"
            "Improved summary:"
        )
        refined = generate_summary("PEGASUS", correction_input)

        # Evaluate refined summary's coverage
        new_nc, new_ec, _, _, _ = compute_graph_coverage(
            G, centrality_scores, refined, anchor_prototype
        )
        iter_log["new_node_coverage"] = round(new_nc, 4)
        iter_log["new_edge_coverage"] = round(new_ec, 4)

        # Accept if either coverage metric improved
        if new_nc > node_cov or new_ec > edge_cov:
            current_summary   = refined
            iter_log["action"] = (
                f"ACCEPTED — node:{node_cov:.3f}→{new_nc:.3f}, "
                f"edge:{edge_cov:.3f}→{new_ec:.3f}"
            )
        else:
            iter_log["action"] = (
                f"REJECTED — node:{new_nc:.3f}≤{node_cov:.3f}, "
                f"edge:{new_ec:.3f}≤{edge_cov:.3f}"
            )

        refine_log["iterations"].append(iter_log)

    # Final coverage stats
    final_nc, final_ec, _, _, _ = compute_graph_coverage(
        G, centrality_scores, current_summary, anchor_prototype
    )
    refine_log["final_node_coverage"] = round(final_nc, 4)
    refine_log["final_edge_coverage"] = round(final_ec, 4)
    return current_summary, refine_log


# =========================================================
# ★ IDEA 12 — EDGE-DIRECTION FACTUAL CONSISTENCY CHECK
# =========================================================

def check_factual_consistency_via_graph(summary_text, G, edge_meta):
    """
    Basic factual consistency check using KG edge directions.

    For each important typed edge (u → v, edge_type) in the KG:
      - Check if both endpoints appear in the summary
      - Check if the relation direction seems preserved (heuristic)

    Example: if KG has "supreme court → SET_ASIDE → high court order"
             but summary says "high court order was upheld", flag inconsistency.

    Returns:
        consistent_edges:   int
        inconsistent_edges: list of (u, v, edge_type, issue)
        consistency_score:  float [0,1]
    """
    summary_lower  = summary_text.lower()
    summary_sents  = sent_tokenize(summary_text)

    # Keyword sets per edge type for direction validation
    EDGE_TYPE_CONFIRM_WORDS = {
        "SET_ASIDE":      ["set aside","quashed","reversed","overturned"],
        "RELIED_ON":      ["relied","followed","applied","cited"],
        "CONVICTED_UNDER":["convicted","held guilty","found guilty"],
        "GRANTED":        ["allowed","granted","upheld","accepted"],
        "REJECTED":       ["dismissed","rejected","denied","refused"],
        "APPEALED_AGAINST":["appeal","challenged","aggrieved"],
    }
    EDGE_TYPE_CONTRADICT_WORDS = {
        "SET_ASIDE":  ["upheld","confirmed","affirmed"],
        "GRANTED":    ["dismissed","rejected","denied"],
        "REJECTED":   ["allowed","granted","upheld"],
    }

    consistent    = 0
    inconsistent  = []
    checked       = 0

    for (u, v), meta in edge_meta.items():
        etype = meta["edge_type"]
        if etype == "RELATED_TO": continue  # generic edges skip
        if u not in summary_lower or v not in summary_lower: continue
        checked += 1

        # Check confirmation words
        confirm_words   = EDGE_TYPE_CONFIRM_WORDS.get(etype, [])
        contradict_words= EDGE_TYPE_CONTRADICT_WORDS.get(etype, [])

        confirmed      = any(w in summary_lower for w in confirm_words)
        contradicted   = any(w in summary_lower for w in contradict_words)

        if contradicted and not confirmed:
            inconsistent.append({
                "u":     u, "v": v, "edge_type": etype,
                "issue": f"Direction possibly reversed — found contradiction words"
            })
        else:
            consistent += 1

    consistency_score = consistent / max(checked, 1) if checked > 0 else 1.0
    return consistent, inconsistent, consistency_score


# =========================================================
# LCS UTILITY FUNCTIONS
# =========================================================

def compute_token_lcs_length(ta, tb):
    if not ta or not tb: return 0
    if len(ta) < len(tb): ta, tb = tb, ta
    n = len(tb)
    prev = [0]*(n+1); curr = [0]*(n+1)
    for a in ta:
        for j, b in enumerate(tb):
            curr[j+1] = prev[j]+1 if a.lower()==b.lower() else max(curr[j],prev[j+1])
        prev, curr = curr, [0]*(n+1)
    return prev[n]

def compute_lcs_score(sentence, source_sentences):
    if not source_sentences: return 0.0
    st = word_tokenize(sentence.lower())
    if not st: return 0.0
    best = 0.0
    for src in source_sentences:
        sr = word_tokenize(src.lower())
        if not sr: continue
        lcs = compute_token_lcs_length(st, sr)
        best = max(best, lcs / max(len(st), len(sr)))
    return best

def find_source_position(sentence, doc_annotation):
    best_pos, best_score = 999, -1.0
    st = word_tokenize(sentence.lower())
    for s in doc_annotation["sentences"]:
        sr = word_tokenize(s["sentence"].lower())
        lcs = compute_token_lcs_length(st, sr)
        if lcs > best_score:
            best_score = lcs; best_pos = s["index"]
    return best_pos


# =========================================================
# NOVEL IDEA 8 — LCS-GUIDED SENTENCE FUSION
# =========================================================

def fuse_adjacent(a, b, min_ngram=LCS_MIN_NGRAM_OVERLAP):
    ta = word_tokenize(a.lower()); tb = word_tokenize(b.lower())
    mx = min(15, len(ta), len(tb))
    for n in range(mx, min_ngram-1, -1):
        if ta[-n:] == tb[:n]:
            bw = word_tokenize(b)
            fw = bw[n:]
            if fw:
                fw[0] = fw[0].lower()
                return f"{a.rstrip('. ')}, {' '.join(fw)}"
    return f"{a} {b}"

def inject_connectives(sentences):
    if not sentences: return sentences
    triggers = {"it","this","they","the same","such","these","those"}
    result = [sentences[0]]; ci = 0
    for sent in sentences[1:]:
        fw = sent.split()[0].lower() if sent.split() else ""
        if fw in triggers:
            conn = LCS_CONNECTIVES[ci % len(LCS_CONNECTIVES)]; ci += 1
            sent = f"{conn} {sent[0].lower()}{sent[1:]}" if conn.endswith("that") \
                   else f"{conn} {sent}"
        result.append(sent)
    return result

def lcs_guided_sentence_fusion(kg_refined_summary, doc_annotation, normalized_weights):
    anchor_sents = [s["sentence"] for s in doc_annotation["sentences"]
                    if s["role"] in LCS_ANCHOR_ROLES]
    all_sents    = [s["sentence"] for s in doc_annotation["sentences"]]
    if not anchor_sents: anchor_sents = all_sents

    summ_sents = sent_tokenize(kg_refined_summary)
    if not summ_sents:
        return kg_refined_summary, {"early_exit": True}

    se = embed_with_legalbert(summ_sents)
    ae = embed_with_legalbert(anchor_sents)
    sm = cosine_similarity(se, ae).max(axis=1)

    scored = []
    for i, sent in enumerate(summ_sents):
        lcs_s = compute_lcs_score(sent, anchor_sents)
        sem_s = float(sm[i])
        scored.append({
            "sentence": sent,
            "score":    LCS_ANCHOR_LCS_WEIGHT*lcs_s + LCS_ANCHOR_SEM_WEIGHT*sem_s,
        })

    selected     = sorted(scored, key=lambda x: x["score"], reverse=True)[:LCS_MAX_OUTPUT_SENTENCES]
    for item in selected:
        item["pos"] = find_source_position(item["sentence"], doc_annotation)
    ordered      = [x["sentence"] for x in sorted(selected, key=lambda x: x["pos"])]

    fused = [ordered[0]]
    for s in ordered[1:]:
        merged = fuse_adjacent(fused[-1], s)
        if merged != f"{fused[-1]} {s}":
            fused[-1] = merged
        else:
            fused.append(s)

    final = inject_connectives(fused)
    return " ".join(final), {"fusions": len(fused), "final_count": len(final)}


# =========================================================
# NOVEL IDEA 9 — VERBATIM SPAN INJECTION
# =========================================================

def _find_sublist(needle, haystack):
    n, h = len(needle), len(haystack)
    if n > h: return -1
    for i in range(h-n+1):
        if haystack[i:i+n] == needle: return i
    return -1

def _find_longest_verbatim_span(sent_tokens, src_tokens, min_span=VERBATIM_MIN_SPAN):
    mw = min(20, len(src_tokens), len(sent_tokens))
    for sl in range(mw, min_span-1, -1):
        for ss in range(len(src_tokens)-sl+1):
            span = src_tokens[ss:ss+sl]
            if _find_sublist(span, sent_tokens) != -1:
                return sl, ss, _find_sublist(span, sent_tokens)
    return 0, -1, -1

def _extract_verbatim_excerpt(src_sent, src_orig, src_start, span_len,
                               sent_len, max_ctx=VERBATIM_MAX_CONTEXT):
    left  = max(0, src_start - max_ctx)
    right = min(len(src_orig), src_start + span_len + max_ctx)
    if (right-left)/max(len(src_orig),1) >= 0.85: return src_sent
    ex = " ".join(src_orig[left:right])
    if ex:
        ex = ex[0].upper() + ex[1:]
        if not ex.endswith("."): ex += "."
    return ex

def inject_verbatim_spans(lcs_fused_summary, doc_annotation):
    critical_src = []
    for s in doc_annotation["sentences"]:
        if s["role"] in VERBATIM_TARGET_ROLES:
            ot = word_tokenize(s["sentence"])
            lt = [t.lower() for t in ot]
            critical_src.append({"sentence":s["sentence"],"role":s["role"],
                                  "orig":ot,"lower":lt})
    if not critical_src:
        return lcs_fused_summary, {"substituted": 0}

    pool       = [s["sentence"] for s in critical_src]
    summ_sents = sent_tokenize(lcs_fused_summary)
    result     = []
    substituted= 0

    for sent in summ_sents:
        sl = word_tokenize(sent.lower())
        if not sl: result.append(sent); continue

        best_span, best_src, best_ss = 0, None, -1
        for src in critical_src:
            sp, ss, _ = _find_longest_verbatim_span(sl, src["lower"])
            if sp > best_span:
                best_span = sp; best_src = src; best_ss = ss

        if best_span < VERBATIM_MIN_SPAN or best_src is None:
            result.append(sent); continue

        lr = len(best_src["lower"]) / max(len(sl), 1)
        if lr > VERBATIM_MAX_LENGTH_RATIO:
            candidate = _extract_verbatim_excerpt(
                best_src["sentence"], best_src["orig"],
                best_ss, best_span, len(sl)
            )
            if len(word_tokenize(candidate.lower()))/max(len(sl),1) > VERBATIM_MAX_LENGTH_RATIO:
                result.append(sent); continue
        else:
            candidate = best_src["sentence"]

        o_lcs = compute_lcs_score(sent,      pool)
        c_lcs = compute_lcs_score(candidate, pool)
        if c_lcs > o_lcs:
            result.append(candidate); substituted += 1
        else:
            result.append(sent)

    return " ".join(result), {"substituted": substituted, "total": len(summ_sents)}


# =========================================================
# STANDARD SUMMARY GENERATION (no KG-Attention)
# =========================================================

def generate_summary(model_name, filtered_text):
    model     = models[model_name]
    tokenizer = tokenizers[model_name]
    config    = MODEL_CONFIG[model_name]
    inputs    = tokenizer(filtered_text, return_tensors="pt",
                          truncation=True,
                          max_length=config["max_input"]).to(device)
    with torch.no_grad():
        if model_name == "LED":
            gam = torch.zeros_like(inputs["attention_mask"])
            gam[:, 0] = 1
            ids = model.generate(
                inputs["input_ids"],
                attention_mask=inputs["attention_mask"],
                global_attention_mask=gam,
                max_length=config["max_output"],
                num_beams=config["num_beams"],
                early_stopping=True, no_repeat_ngram_size=3,
            )
        else:
            ids = model.generate(
                inputs["input_ids"],
                attention_mask=inputs["attention_mask"],
                max_length=config["max_output"],
                num_beams=config["num_beams"],
                early_stopping=True, no_repeat_ngram_size=3,
            )
    return tokenizer.decode(ids[0], skip_special_tokens=True)


# =========================================================
# NOVEL IDEA 11 — STRUCTURE-AWARE SELECTIVE PARAPHRASING
# =========================================================

def get_legal_anchor_prototype():
    global _LEGAL_ANCHOR_PROTOTYPE
    if _LEGAL_ANCHOR_PROTOTYPE is not None:
        return _LEGAL_ANCHOR_PROTOTYPE
    embs = embed_with_legalbert(LEGAL_ANCHOR_SEED_TOKENS)
    _LEGAL_ANCHOR_PROTOTYPE = embs.mean(0, keepdims=True) if embs.shape[0] > 0 \
                              else np.zeros((1,768))
    return _LEGAL_ANCHOR_PROTOTYPE

def detect_legal_anchors_regex(sentence):
    spans, texts, seen = [], [], set()
    for pat in LEGAL_ANCHOR_PATTERNS:
        for m in pat.finditer(sentence):
            sp = (m.start(), m.end())
            if not any(s <= sp[0] < e or s < sp[1] <= e for s,e in seen):
                spans.append(sp); texts.append(m.group().strip()); seen.add(sp)
    paired = sorted(zip(spans, texts), key=lambda x: x[0][0])
    if paired:
        spans, texts = zip(*paired)
        return list(spans), list(texts)
    return [], []

@torch.no_grad()
def detect_legal_anchors_contextual(words, anchor_proto, threshold=LEGAL_ANCHOR_THRESHOLD):
    cands = [w for w in words if len(w) >= 4]
    if not cands: return [False]*len(words)
    embs = embed_with_legalbert(cands)
    if embs.shape[0] == 0: return [False]*len(words)
    sims = cosine_similarity(embs, anchor_proto).squeeze()
    if sims.ndim == 0: sims = np.array([float(sims)])
    anchor_map = {}; ci = 0
    for w in words:
        anchor_map[w] = (float(sims[ci]) >= threshold if ci < len(sims) else False) \
                        if len(w) >= 4 else False
        if len(w) >= 4: ci += 1
    return [anchor_map.get(w, False) for w in words]

def detect_all_legal_anchors(sentence, anchor_proto):
    words = word_tokenize(sentence)
    if not words: return [], [], []

    regex_spans, anchor_texts = detect_legal_anchors_regex(sentence)
    regex_flags = [False]*len(words)
    search_pos  = 0
    word_positions = []
    for w in words:
        pos = sentence.find(w, search_pos)
        if pos == -1: pos = search_pos
        word_positions.append((pos, pos+len(w)))
        search_pos = pos+1

    for (ss, se) in regex_spans:
        for wi, (ws, we) in enumerate(word_positions):
            if ws < se and we > ss:
                regex_flags[wi] = True

    unanchored = [w for w, f in zip(words, regex_flags) if not f]
    if unanchored:
        ctx_sub = detect_legal_anchors_contextual(unanchored, anchor_proto)
        ctx_flags = []
        sub_idx = 0
        for f in regex_flags:
            if f:   ctx_flags.append(False)
            else:   ctx_flags.append(ctx_sub[sub_idx] if sub_idx < len(ctx_sub) else False); sub_idx += 1
    else:
        ctx_flags = [False]*len(words)

    combined = [a or b for a, b in zip(regex_flags, ctx_flags)]
    return combined, anchor_texts, words

def strip_ceremony_prefix(sentence):
    for pat in CEREMONY_PATTERNS:
        m = pat.match(sentence)
        if m:
            stripped = sentence[m.end():].strip()
            if stripped: stripped = stripped[0].upper() + stripped[1:]
            return stripped, True
    return sentence, False

def split_into_runs(words, flags):
    if not words: return []
    runs   = []
    ctype  = 'anchor' if flags[0] else 'prose'
    ctoks  = [words[0]]
    for w, f in zip(words[1:], flags[1:]):
        wt = 'anchor' if f else 'prose'
        if wt == ctype: ctoks.append(w)
        else:
            runs.append({"type":ctype, "tokens":ctoks})
            ctype = wt; ctoks = [w]
    runs.append({"type":ctype,"tokens":ctoks})
    return runs

def paraphrase_prose(prose_text, pegasus_model, pegasus_tokenizer,
                     aggressiveness=0.5, max_length=128):
    if not prose_text.strip() or len(prose_text.split()) < 3:
        return prose_text
    if aggressiveness < MIN_AGGR_TO_PARAPHRASE:
        stripped, _ = strip_ceremony_prefix(prose_text)
        return stripped if stripped else prose_text
    try:
        inp = pegasus_tokenizer(
            PARAPHRASE_INSTRUCTION + prose_text,
            return_tensors="pt", truncation=True, max_length=512
        ).to(device)
        with torch.no_grad():
            out_ids = pegasus_model.generate(
                inp["input_ids"], attention_mask=inp["attention_mask"],
                max_length=max_length, num_beams=2,
                early_stopping=True, no_repeat_ngram_size=2, length_penalty=0.8,
            )
        para = pegasus_tokenizer.decode(out_ids[0], skip_special_tokens=True).strip()
        if para and len(para.split()) <= len(prose_text.split())*(1+aggressiveness*0.3):
            return para
    except Exception:
        pass
    return prose_text

def reconstruct_from_runs(runs, prose_map):
    parts = []
    for i, run in enumerate(runs):
        parts.append(prose_map[i] if run["type"]=="prose" and i in prose_map
                     else " ".join(run["tokens"]))
    sent = " ".join(p.strip() for p in parts if p.strip())
    sent = re.sub(r'\s+([,\.\;\:\!\?])', r'\1', sent)
    return re.sub(r'\s+', ' ', sent).strip()

def get_sentence_role_from_doc(sentence, doc_annotation, anchor_proto):
    st = word_tokenize(sentence.lower())
    best_role, best_lcs = "REASONING", -1
    for s in doc_annotation["sentences"]:
        sr  = word_tokenize(s["sentence"].lower())
        lcs = compute_token_lcs_length(st, sr)
        if lcs > best_lcs: best_lcs = lcs; best_role = s["role"]
    if best_lcs < 3:
        se  = embed_with_legalbert([sentence])
        ss  = [s["sentence"] for s in doc_annotation["sentences"]]
        if ss:
            se2  = embed_with_legalbert(ss)
            sims = cosine_similarity(se, se2).squeeze()
            best_role = doc_annotation["sentences"][int(np.argmax(sims))]["role"]
    return best_role

def canonical_reconstruction(paraphrased, roles, doc_annotation):
    items = []
    for sent, role in zip(paraphrased, roles):
        items.append({
            "sentence":     sent,
            "canonical_pos":CANONICAL_POSITION.get(role, 7),
            "source_pos":   find_source_position(sent, doc_annotation),
            "role":         role,
        })
    items.sort(key=lambda x: (x["canonical_pos"], x["source_pos"]))
    return [x["sentence"] for x in items]

def structure_aware_selective_paraphrase(verbatim_summary, doc_annotation,
                                          pegasus_model, pegasus_tokenizer,
                                          reference_pool=None):
    anchor_proto  = get_legal_anchor_prototype()
    if reference_pool is None:
        reference_pool = [s["sentence"] for s in doc_annotation["sentences"]]

    summ_sents = sent_tokenize(verbatim_summary)
    if not summ_sents: return verbatim_summary, {"total":0}

    paraphrased = []
    roles       = []
    log         = {"total":len(summ_sents),"paraphrased":0,"skipped":0,
                   "lcs_rejected":0,"ceremony_stripped":0}

    for sent in summ_sents:
        role  = get_sentence_role_from_doc(sent, doc_annotation, anchor_proto)
        aggr  = ROLE_PARAPHRASE_AGGR.get(role, 0.3)
        roles.append(role)

        if aggr is None:
            paraphrased.append(sent); log["skipped"] += 1; continue

        anchor_flags, _, words = detect_all_legal_anchors(sent, anchor_proto)
        if not words: paraphrased.append(sent); continue

        anchor_ratio = sum(anchor_flags) / max(len(words), 1)
        if anchor_ratio >= 0.70:
            paraphrased.append(sent); log["skipped"] += 1; continue

        working, ceremony_stripped = strip_ceremony_prefix(sent)
        if ceremony_stripped:
            log["ceremony_stripped"] += 1
            anchor_flags, _, words = detect_all_legal_anchors(working, anchor_proto)

        runs = split_into_runs(words, anchor_flags)
        prose_map = {}
        for i, run in enumerate(runs):
            if run["type"] == "prose" and len(run["tokens"]) >= 3:
                para = paraphrase_prose(
                    " ".join(run["tokens"]), pegasus_model,
                    pegasus_tokenizer, aggr
                )
                if para != " ".join(run["tokens"]):
                    prose_map[i] = para

        candidate = reconstruct_from_runs(runs, prose_map) if prose_map else working

        orig_lcs = compute_lcs_score(sent,      reference_pool)
        cand_lcs = compute_lcs_score(candidate, reference_pool)

        if cand_lcs >= orig_lcs - LCS_REGRESSION_TOLERANCE:
            paraphrased.append(candidate); log["paraphrased"] += 1
        else:
            paraphrased.append(sent); log["lcs_rejected"] += 1

    reordered = canonical_reconstruction(paraphrased, roles, doc_annotation)
    return " ".join(reordered), log


# =========================================================
# ROLE ANNOTATION PIPELINE
# =========================================================

def create_role_annotations(data, output_file):
    print(f"\n{'='*65}")
    print("STEP 1: SENTENCE-ROLE ANNOTATIONS (8 LABELS)")
    print(f"{'='*65}")
    annotations = []
    for idx, item in enumerate(tqdm(data, desc="Annotating")):
        judgment  = item.get("judgment","")
        doc_id    = item.get("id", idx)
        sentences = sent_tokenize(judgment)
        if not sentences: continue
        roles = classify_roles(sentences)
        doc   = {
            "doc_id":          doc_id,
            "total_sentences": len(sentences),
            "sentences":       [{"index":i,"sentence":s,"role":r}
                                 for i,(s,r) in enumerate(zip(sentences,roles))],
        }
        annotations.append(doc)
    with open(output_file,'w',encoding='utf8') as f:
        json.dump(annotations, f, indent=2, ensure_ascii=False)
    print(f"✅  Annotations saved → {output_file} ({len(annotations)} docs)")
    _print_role_stats(annotations)
    return annotations

def _print_role_stats(annotations):
    rc = Counter(); total = 0
    for doc in annotations:
        for s in doc["sentences"]:
            rc[s["role"]] += 1; total += 1
    print(f"\n{'Role':<28} {'Count':>7} {'%':>7}  Bar")
    print("-"*60)
    for role in LABELS_8:
        c   = rc[role]
        pct = c/total*100 if total else 0
        print(f"  {role:<26} {c:>7} {pct:>6.2f}%  {'█'*int(pct/2)}")
    print("-"*60)
    print(f"  {'TOTAL':<26} {total:>7}")

def calculate_role_contribution(train_ann, train_data, output_file):
    print(f"\n{'='*65}")
    print("STEP 2: ROLE CONTRIBUTION SCORES")
    print(f"{'='*65}")
    rtc = Counter(); rsc = Counter()
    for ann, item in tqdm(zip(train_ann, train_data), total=len(train_ann),
                           desc="Computing"):
        ref  = item.get("reference_summary","")
        rsents = sent_tokenize(ref)
        dsents = [s["sentence"] for s in ann["sentences"]]
        droles = [s["role"]     for s in ann["sentences"]]
        for r in droles: rtc[r] += 1
        if dsents and rsents:
            de = embed_with_legalbert(dsents)
            re_ = embed_with_legalbert(rsents)
            for rem in re_:
                sims = cosine_similarity([rem], de)[0]
                bi   = np.argmax(sims)
                if sims[bi] > 0.7: rsc[droles[bi]] += 1
    scores = {r: rsc[r]/rtc[r] if rtc[r] > 0 else 0.0 for r in LABELS_8}
    data   = {"role_scores": scores, "role_total_counts": dict(rtc),
              "role_summary_counts": dict(rsc), "mapping_used": LABEL_MAPPING_13_TO_8}
    with open(output_file,'w',encoding='utf8') as f:
        json.dump(data, f, indent=2, ensure_ascii=False)
    print(f"✅  Contribution scores saved → {output_file}")
    return scores

def normalize_role_weights(role_scores, output_file):
    total = sum(role_scores.values())
    nw    = ({r: s/total for r, s in role_scores.items()} if total > 0
             else {r: 1/len(LABELS_8) for r in LABELS_8})
    with open(output_file,'w',encoding='utf8') as f:
        json.dump({"normalized_weights":nw,"original_scores":role_scores}, f, indent=2)
    print(f"\n✅  Normalized weights saved → {output_file}")
    for r, w in sorted(nw.items(), key=lambda x: x[1], reverse=True):
        print(f"   {r:<28}: {w:.4f}  ({w*100:.1f}%)")
    return nw


# =========================================================
# ENSEMBLE STRATEGIES
# =========================================================

def ensemble_voting(summaries_dict):
    all_s   = []
    for s in summaries_dict.values(): all_s.extend(sent_tokenize(s))
    counts  = Counter(s.lower().strip() for s in all_s)
    sel     = [s for s,c in counts.items() if c >= 2]
    if len(sel) < 3: sel = [s for s,_ in counts.most_common(10)]
    return " ".join(sel)

def ensemble_weighted_concat(summaries_dict, weights):
    parts = []
    for m, s in summaries_dict.items():
        ss = sent_tokenize(s)
        parts.extend(ss[:max(1, int(len(ss)*weights.get(m, 0.25)))])
    seen = set(); out = []
    for s in parts:
        if s.lower().strip() not in seen:
            seen.add(s.lower().strip()); out.append(s)
    return " ".join(out)

def ensemble_best_model(summaries_dict, reference):
    best_s, best_m = -1, ""
    for m, s in summaries_dict.items():
        sc = rouge_metric.compute(predictions=[s], references=[reference],
                                   use_stemmer=True)["rouge2"]
        if sc > best_s: best_s = sc; best_m = s
    return best_m

def ensemble_sentence_ranking(summaries_dict):
    pos = {}
    for m, s in summaries_dict.items():
        for i, sent in enumerate(sent_tokenize(s)):
            k = sent.lower().strip()
            if k not in pos: pos[k] = []
            pos[k].append(i)
    ranked = sorted(pos.items(), key=lambda x: np.mean(x[1]))
    return " ".join(s for s,_ in ranked[:15])


# =========================================================
# LOAD SUMMARIZATION MODELS
# =========================================================

print("\n📚 Loading summarization models...")
models = {}; tokenizers = {}

for name, cls, tok_cls in [
    ("BART",    AutoModelForSeq2SeqLM,          AutoTokenizer),
    ("LED",     LEDForConditionalGeneration,     AutoTokenizer),
    ("PEGASUS", PegasusForConditionalGeneration, AutoTokenizer),
]:
    print(f"  Loading {name}...")
    tokenizers[name] = tok_cls.from_pretrained(MODEL_PATHS[name])
    models[name]     = cls.from_pretrained(MODEL_PATHS[name]).to(device)
    models[name].eval()
    print(f"  ✅  {name} loaded")

# Pre-compute legal anchor prototype
print("\n🔧 Pre-computing legal anchor prototype...")
get_legal_anchor_prototype()
print("  ✅  Legal anchor prototype ready")

# =========================================================
# EVALUATION
# =========================================================

print("\n📊 Loading evaluation metrics...")
rouge_metric     = evaluate.load("rouge")
bertscore_metric = evaluate.load("bertscore")

def compute_metrics(predictions, references):
    rouge = rouge_metric.compute(predictions=predictions, references=references,
                                  use_stemmer=True)
    bert  = bertscore_metric.compute(predictions=predictions, references=references,
                                      model_type="roberta-large", lang="en",
                                      device=device, batch_size=8)
    return {
        "rouge1":       float(rouge["rouge1"]),
        "rouge2":       float(rouge["rouge2"]),
        "rougeL":       float(rouge["rougeL"]),
        "bertscore_f1": float(np.mean(bert["f1"])),
    }


# =========================================================
# MAIN PIPELINE
# =========================================================

def main():
    print("\n" + "="*70)
    print("🚀  ENSEMBLE LEGAL SUMMARIZATION v4 — REAL KNOWLEDGE GRAPH")
    print("="*70)
    print("   Idea 5+7: KG-Diff + Semantic Coverage")
    print("   Idea 8  : LCS-Guided Sentence Fusion")
    print("   Idea 9  : Verbatim Span Injection")
    print("   Idea 10 : KG-Attention (centrality-proportional)")
    print("   Idea 11 : Structure-Aware Selective Paraphrasing")
    print("   ★ Idea 12: Real Per-Document Legal Knowledge Graph")
    print("            NetworkX DiGraph | Typed nodes+edges | PageRank")
    print("            Betweenness | Degree | Graph Coverage KG-Diff")
    print("            Centrality-Proportional Entity Attention")
    print("            Edge-Direction Factual Consistency Check")
    print("="*70)

    # ── Load data ──────────────────────────────────────────────────────────
    print(f"\n📂 Loading training data ({MAX_TRAIN_SAMPLES} samples max)...")
    with open(TRAIN_JSON_PATH,'r',encoding='utf8') as f:
        train_data = json.load(f)[:MAX_TRAIN_SAMPLES]
    print(f"   {len(train_data)} training samples loaded")

    print(f"\n📂 Loading test data...")
    with open(TEST_JSON_PATH,'r',encoding='utf8') as f:
        test_data = json.load(f)
    print(f"   {len(test_data)} test samples loaded")

    # ── Steps 1-3: Annotations → Contribution → Weights ───────────────────
    ann_path  = os.path.join(OUTPUT_DIR, ROLE_CLASSIFICATION_FILE)
    cont_path = os.path.join(OUTPUT_DIR, ROLE_CONTRIBUTION_FILE)
    wt_path   = os.path.join(OUTPUT_DIR, ROLE_WEIGHTS_FILE)

    if os.path.exists(ann_path):
        with open(ann_path,'r',encoding='utf8') as f: train_ann = json.load(f)
        print(f"\n📂 Loaded existing train annotations ({len(train_ann)} docs)")
    else:
        train_ann = create_role_annotations(train_data, ann_path)

    if os.path.exists(cont_path):
        with open(cont_path,'r',encoding='utf8') as f: role_scores = json.load(f)["role_scores"]
    else:
        role_scores = calculate_role_contribution(train_ann, train_data, cont_path)

    if os.path.exists(wt_path):
        with open(wt_path,'r',encoding='utf8') as f: nw = json.load(f)["normalized_weights"]
    else:
        nw = normalize_role_weights(role_scores, wt_path)

    # ── Step 4: Test annotations ───────────────────────────────────────────
    test_ann_path = os.path.join(OUTPUT_DIR, "test_"+ROLE_CLASSIFICATION_FILE)
    if os.path.exists(test_ann_path):
        with open(test_ann_path,'r',encoding='utf8') as f: test_ann = json.load(f)
        print(f"\n📂 Loaded existing test annotations ({len(test_ann)} docs)")
    else:
        test_ann = create_role_annotations(test_data, test_ann_path)

    # ── Step 5: Generate + evaluate ───────────────────────────────────────
    print(f"\n{'='*70}")
    print("STEP 5: GENERATING SUMMARIES — FULL PIPELINE v4")
    print(f"{'='*70}")

    all_sums = {m: [] for m in ["BART","LED","PEGASUS","LED_kg_cent"]}
    ens_sums = {k: [] for k in [
        "voting","weighted","best_model","ranking",
        "kg_refined","lcs_fused","verbatim_injected",
        "kg_graph_refined",        # ★ Idea 12: Graph Coverage KG-Diff
        "structure_paraphrase",    # Idea 11
        "full_stack",              # Full pipeline: KG→Graph→LCS→Verbatim→Paraphrase
    ]}

    references   = []
    kg_build_logs= []   # ★ KG diagnostics
    fact_logs    = []   # ★ Factual consistency logs
    anchor_proto = get_legal_anchor_prototype()

    print("\n🔄 Processing documents...")
    for ann, item in tqdm(zip(test_ann, test_data), total=len(test_data),
                           desc="Documents"):
        ref = item["reference_summary"]
        references.append(ref)

        targets = get_adaptive_targets(ann["total_sentences"])

        # ── ★ IDEA 12: Build Real Legal KG ────────────────────────────────
        G, node_meta, edge_meta, kg_log = build_legal_kg(
            ann, roles_to_include=None   # include all roles
        )
        centrality_scores, raw_centrality = compute_graph_centrality(G)
        kg_log["centrality_computed"] = len(centrality_scores)

        if len(kg_build_logs) < 3:
            kg_build_logs.append({"doc_id": ann["doc_id"], "kg_log": kg_log,
                                   "top_nodes_by_centrality": sorted(
                                       raw_centrality.items(),
                                       key=lambda x: x[1]["unified"], reverse=True
                                   )[:10]})

        # ── Standard generation (BART, PEGASUS) ───────────────────────────
        summ_dict = {}
        for m in ["BART", "PEGASUS"]:
            ft, _ = hybrid_role_salience_selection(ann, nw, targets[m])
            s = generate_summary(m, ft)
            all_sums[m].append(s); summ_dict[m] = s

        # ── Standard LED ──────────────────────────────────────────────────
        ft_led, _ = hybrid_role_salience_selection(ann, nw, targets["LED"])
        led_std   = generate_summary("LED", ft_led)
        all_sums["LED"].append(led_std); summ_dict["LED"] = led_std

        # ── ★ IDEA 12: Centrality-Guided Selection + KG-Attention LED ─────
        ft_kg, sel_info = graph_centrality_guided_selection(
            ann, nw, G, centrality_scores, targets["LED"]
        )
        led_kg_cent = generate_summary_kg_centrality_attention(
            "LED", ft_kg, G, centrality_scores
        )
        all_sums["LED_kg_cent"].append(led_kg_cent)
        summ_dict["LED_kg_cent"] = led_kg_cent

        # ── Basic ensembles ────────────────────────────────────────────────
        std3 = {k: summ_dict[k] for k in ["BART","LED","PEGASUS"]}
        ens_sums["voting"].append(ensemble_voting(std3))
        ens_sums["weighted"].append(ensemble_weighted_concat(std3, ENSEMBLE_WEIGHTS))
        ens_sums["best_model"].append(ensemble_best_model(std3, ref))
        ens_sums["ranking"].append(ensemble_sentence_ranking(std3))

        # ── Original pseudo-KG refinement (Idea 5+7 baseline) ─────────────
        # Use standard cosine triple coverage for comparison
        from collections import namedtuple
        kg_refined, _ = _pseudo_kg_refinement(summ_dict, ann)
        ens_sums["kg_refined"].append(kg_refined)

        # ── LCS Fusion (Idea 8) ────────────────────────────────────────────
        lcs_fused, _ = lcs_guided_sentence_fusion(kg_refined, ann, nw)
        ens_sums["lcs_fused"].append(lcs_fused)

        # ── Verbatim Injection (Idea 9) ────────────────────────────────────
        verbatim, _ = inject_verbatim_spans(lcs_fused, ann)
        ens_sums["verbatim_injected"].append(verbatim)

        # ── ★ IDEA 12: Graph Coverage KG-Diff Refinement ──────────────────
        kg_graph_summaries = {
            "BART":         summ_dict["BART"],
            "LED":          summ_dict["LED_kg_cent"],
            "PEGASUS":      summ_dict["PEGASUS"],
            "LED_kg_attn":  summ_dict["LED_kg_cent"],
        }
        kg_graph_refined, refine_log = kg_graph_coverage_refinement(
            kg_graph_summaries, ann, G, centrality_scores, anchor_proto
        )
        ens_sums["kg_graph_refined"].append(kg_graph_refined)

        # ── ★ IDEA 12: Factual Consistency Check ──────────────────────────
        consistent, inconsistent, fc_score = check_factual_consistency_via_graph(
            kg_graph_refined, G, edge_meta
        )
        if len(fact_logs) < 3:
            fact_logs.append({
                "doc_id":            ann["doc_id"],
                "consistent_edges":  consistent,
                "inconsistent":      inconsistent[:3],
                "consistency_score": round(fc_score, 4),
            })

        # ── Idea 11: Structure-Aware Paraphrase (on verbatim output) ──────
        struct_para, _ = structure_aware_selective_paraphrase(
            verbatim, ann,
            models["PEGASUS"], tokenizers["PEGASUS"],
            reference_pool=[s["sentence"] for s in ann["sentences"]]
        )
        ens_sums["structure_paraphrase"].append(struct_para)

        # ── Full Stack: KG Graph Refined → LCS → Verbatim → Paraphrase ────
        fs_lcs,  _  = lcs_guided_sentence_fusion(kg_graph_refined, ann, nw)
        fs_verb, _  = inject_verbatim_spans(fs_lcs, ann)
        fs_para, _  = structure_aware_selective_paraphrase(
            fs_verb, ann,
            models["PEGASUS"], tokenizers["PEGASUS"],
            reference_pool=[s["sentence"] for s in ann["sentences"]]
        )
        ens_sums["full_stack"].append(fs_para)

    print("✅  All summaries generated!")

    # ── Save KG and factual consistency logs ──────────────────────────────
    with open(os.path.join(OUTPUT_DIR,"kg_build_logs.json"),'w',encoding='utf8') as f:
        json.dump(kg_build_logs, f, indent=2, ensure_ascii=False)
    with open(os.path.join(OUTPUT_DIR,"factual_consistency_logs.json"),'w',encoding='utf8') as f:
        json.dump(fact_logs, f, indent=2, ensure_ascii=False)
    print(f"\n📊 KG + factual logs saved to: {OUTPUT_DIR}/")

    # ── Evaluation ────────────────────────────────────────────────────────
    print(f"\n{'='*70}")
    print("📊 EVALUATING ALL APPROACHES")
    print(f"{'='*70}")

    results = {}

    for name, sums in [
        ("BART",         all_sums["BART"]),
        ("LED",          all_sums["LED"]),
        ("PEGASUS",      all_sums["PEGASUS"]),
        ("LED_kg_cent ★",all_sums["LED_kg_cent"]),
    ]:
        print(f"\n  Evaluating {name}...")
        m = compute_metrics(sums, references)
        results[name] = m
        tag = " 🎯" if m["rougeL"] >= 0.34 else ""
        print(f"  R1:{m['rouge1']:.4f}  R2:{m['rouge2']:.4f}  "
              f"RL:{m['rougeL']:.4f}{tag}  BS:{m['bertscore_f1']:.4f}")

    strat_labels = {
        "voting":            "Ensemble Voting",
        "weighted":          "Ensemble Weighted",
        "best_model":        "Ensemble Best (oracle)",
        "ranking":           "Ensemble Ranking",
        "kg_refined":        "Pseudo-KG Refined (baseline)",
        "lcs_fused":         "LCS-Fused (Idea 8)",
        "verbatim_injected": "Verbatim Injected (Idea 9)",
        "kg_graph_refined":  "★ Graph Coverage KG-Diff (Idea 12)",
        "structure_paraphrase": "Structure-Aware Paraphrase (Idea 11)",
        "full_stack":        "★ FULL STACK (Ideas 8+9+11+12)",
    }

    for strategy, label in strat_labels.items():
        print(f"\n  Evaluating {label}...")
        m = compute_metrics(ens_sums[strategy], references)
        results[f"ens_{strategy}"] = m
        tag = " 🎯 TARGET!" if m["rougeL"] >= 0.34 else ""
        print(f"  R1:{m['rouge1']:.4f}  R2:{m['rouge2']:.4f}  "
              f"RL:{m['rougeL']:.4f}{tag}  BS:{m['bertscore_f1']:.4f}")

    # ── Final results table ────────────────────────────────────────────────
    print(f"\n{'='*95}")
    print("📊 FINAL RESULTS (sorted by ROUGE-L)")
    print(f"{'='*95}")
    print(f"{'Approach':<48} {'R1':<8} {'R2':<8} {'RL':<8} {'Status':<17} BertF1")
    print("-"*95)

    for approach, m in sorted(results.items(), key=lambda x: x[1]["rougeL"], reverse=True):
        rl     = m["rougeL"]
        status = "✓ ≥0.34" if rl >= 0.34 else f"(-{0.34-rl:.4f})"
        star   = " ★" if ("graph" in approach or "full_stack" in approach
                          or "kg_cent" in approach) else ""
        print(f"{approach+star:<48} {m['rouge1']:<8.4f} {m['rouge2']:<8.4f} "
              f"{rl:<8.4f} {status:<17} {m['bertscore_f1']:.4f}")

    best_rl = max(results.items(), key=lambda x: x[1]["rougeL"])
    best_r2 = max(results.items(), key=lambda x: x[1]["rouge2"])
    best_bs = max(results.items(), key=lambda x: x[1]["bertscore_f1"])
    print(f"\n{'='*95}")
    print(f"🏆 BEST ROUGE-L   : {best_rl[0]} → {best_rl[1]['rougeL']:.4f}")
    print(f"🏆 BEST ROUGE-2   : {best_r2[0]} → {best_r2[1]['rouge2']:.4f}")
    print(f"🏆 BEST BERTScore : {best_bs[0]} → {best_bs[1]['bertscore_f1']:.4f}")
    print(f"{'='*95}")

    # ── Save all summaries ─────────────────────────────────────────────────
    print("\n💾 Saving summaries...")
    idx_map = list(zip(test_data, references))
    for name, sums in all_sums.items():
        path = os.path.join(OUTPUT_DIR, f"{name.lower()}_summaries.json")
        with open(path,'w',encoding='utf8') as f:
            json.dump([{"id":item.get("id",i),"generated":s,"reference":ref}
                       for i,((item,ref),s) in enumerate(zip(idx_map,sums))],
                      f, indent=2, ensure_ascii=False)
    for strategy, sums in ens_sums.items():
        path = os.path.join(OUTPUT_DIR, f"ens_{strategy}_summaries.json")
        with open(path,'w',encoding='utf8') as f:
            json.dump([{"id":item.get("id",i),"generated":s,"reference":ref}
                       for i,((item,ref),s) in enumerate(zip(idx_map,sums))],
                      f, indent=2, ensure_ascii=False)

    # ── Save complete results ──────────────────────────────────────────────
    complete = {
        "experiment": "Ensemble Legal Summarization v4 — Real KG",
        "idea_12_config": {
            "node_types":              list(KG_NODE_TYPES.keys()),
            "edge_types":              list(KG_EDGE_TYPES.keys()),
            "centrality_weights":      KG_CENTRALITY_WEIGHTS,
            "node_coverage_threshold": KG_NODE_COVERAGE_THRESHOLD,
            "edge_coverage_threshold": KG_EDGE_COVERAGE_THRESHOLD,
            "important_node_threshold":KG_IMPORTANT_NODE_THRESHOLD,
            "hybrid_weights": {"alpha_role":HYBRID_ALPHA,
                               "beta_salience":HYBRID_BETA,
                               "gamma_centrality":HYBRID_GAMMA},
        },
        "results":    results,
        "best_rougeL":{"name":best_rl[0],"metrics":best_rl[1]},
        "best_rouge2":{"name":best_r2[0],"metrics":best_r2[1]},
        "best_bertscore":{"name":best_bs[0],"metrics":best_bs[1]},
    }
    rpath = os.path.join(OUTPUT_DIR, "complete_results_v4.json")
    with open(rpath,'w',encoding='utf8') as f:
        json.dump(complete, f, indent=2, ensure_ascii=False)

    print(f"\n✅  All results saved → {OUTPUT_DIR}/")
    print(f"\n{'='*70}")
    print("✅  PIPELINE v4 COMPLETE!")
    print(f"{'='*70}")
    print("\n★  Idea 12 Summary — Real Legal Knowledge Graph:")
    print(f"   Node types:    {list(KG_NODE_TYPES.keys())}")
    print(f"   Edge types:    {list(KG_EDGE_TYPES.keys())}")
    print(f"   Centrality:    PageRank({KG_CENTRALITY_WEIGHTS['pagerank']}) "
          f"+ Betweenness({KG_CENTRALITY_WEIGHTS['betweenness']}) "
          f"+ Degree({KG_CENTRALITY_WEIGHTS['degree']})")
    print(f"   Selection:     α={HYBRID_ALPHA}×role + β={HYBRID_BETA}×salience "
          f"+ γ={HYBRID_GAMMA}×centrality")
    print(f"   KG-Diff:       Graph node+edge coverage (not triple cosine)")
    print(f"   Attention:     Centrality-proportional boost (not flat)")
    print(f"   Fact Check:    Edge-direction consistency validation")
    print(f"\n📊 Full stacked pipeline (full_stack):")
    print("   Centrality-Guided Selection")
    print("   → KG-Attention (centrality-proportional)")
    print("   → Graph Coverage KG-Diff Refinement")
    print("   → LCS-Guided Sentence Fusion")
    print("   → Verbatim Span Injection")
    print("   → Structure-Aware Selective Paraphrasing")
    print("   → Canonical Rhetorical Reconstruction")


# =========================================================
# HELPER: Pseudo-KG refinement (kept as baseline comparison)
# =========================================================

def _pseudo_kg_refinement(summaries_dict, doc_annotation,
                           max_iter=KG_MAX_ITERATIONS):
    """
    Pseudo-KG refinement using triple-cosine coverage (old method).
    Kept as a comparison baseline against the real graph approach.
    """
    critical_sents = [s["sentence"] for s in doc_annotation["sentences"]
                      if s["role"] in KG_CRITICAL_ROLES]
    if not critical_sents:
        return summaries_dict.get("LED",""), {}

    # Extract triples as text
    triples_raw = extract_typed_triples(critical_sents, use_spacy=True)
    if not triples_raw:
        return summaries_dict.get("LED",""), {}

    crit_triple_texts = [f"{t['subj']} {t['rel']} {t['obj']}" for t in triples_raw]
    crit_embs         = embed_with_legalbert(crit_triple_texts)

    current = (summaries_dict.get("LED_kg_attn","") or
               summaries_dict.get("LED",""))

    for _ in range(max_iter):
        summ_sents  = sent_tokenize(current)
        summ_triples= extract_typed_triples(summ_sents)
        if not summ_triples: break

        summ_texts  = [f"{t['subj']} {t['rel']} {t['obj']}" for t in summ_triples]
        summ_embs   = embed_with_legalbert(summ_texts)
        sim_matrix  = cosine_similarity(crit_embs, summ_embs)
        coverage    = (sim_matrix.max(axis=1) >= 0.75).mean()

        if coverage >= 0.80: break

        uncovered   = [crit_triple_texts[i] for i in range(len(crit_embs))
                       if sim_matrix[i].max() < 0.75][:5]
        missing_ents= set(w for t in uncovered for w in t.split() if len(w) > 3)

        corr_sents  = [s["sentence"] for s in doc_annotation["sentences"]
                       if any(e in s["sentence"].lower() for e in missing_ents)][:10]
        if not corr_sents: break

        refined = generate_summary("LED",
            f"Improve this legal summary by including the missing information.\n\n"
            f"Current summary: {current}\n\n"
            f"Missing information: {' '.join(corr_sents)}\n\nImproved summary:")

        # Accept if different and non-empty
        if refined and refined != current:
            current = refined
        else:
            break

    return current, {}


# =========================================================
# ENTRY POINT
# =========================================================

if __name__ == "__main__":
    try:
        main()
    except KeyboardInterrupt:
        print("\n\n⚠️  Pipeline interrupted by user")
    except Exception as e:
        print(f"\n\n❌  Error: {e}")
        import traceback; traceback.print_exc()

🚀  Device: cuda

📚 Loading HSLN role classifier...
✅  HSLN role classifier loaded

📚 Loading InLegalBERT...


/home/RSlab/.conda/envs/legal-nlp/lib/python3.10/site-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


✅  InLegalBERT loaded

📚 Loading summarization models...
  Loading BART...
  ✅  BART loaded
  Loading LED...
  ✅  LED loaded
  Loading PEGASUS...
  ✅  PEGASUS loaded

🔧 Pre-computing legal anchor prototype...
  ✅  Legal anchor prototype ready

📊 Loading evaluation metrics...

🚀  ENSEMBLE LEGAL SUMMARIZATION v4 — REAL KNOWLEDGE GRAPH
   Idea 5+7: KG-Diff + Semantic Coverage
   Idea 8  : LCS-Guided Sentence Fusion
   Idea 9  : Verbatim Span Injection
   Idea 10 : KG-Attention (centrality-proportional)
   Idea 11 : Structure-Aware Selective Paraphrasing
   ★ Idea 12: Real Per-Document Legal Knowledge Graph
            NetworkX DiGraph | Typed nodes+edges | PageRank
            Betweenness | Degree | Graph Coverage KG-Diff
            Centrality-Proportional Entity Attention
            Edge-Direction Factual Consistency Check

📂 Loading training data (3000 samples max)...
   3000 training samples loaded

📂 Loading test data...
   100 test samples loaded

STEP 1: SENTENCE-ROLE ANNOTATIONS 

Annotating: 100%|███████████████████████████████████████████████████████████████████| 3000/3000 [18:39<00:00,  2.68it/s]


✅  Annotations saved → ./ensemble_results_real_kg_v4/sentence_role_annotations_8label.json (3000 docs)

Role                           Count       %  Bar
------------------------------------------------------------
  PREAMBLE                       329   0.08%  
  FACTS                           85   0.02%  
  ISSUE                           28   0.01%  
  ARGUMENTS                   215153  54.61%  ███████████████████████████
  REASONING                    80415  20.41%  ██████████
  PRECEDENT_RELIED               408   0.10%  
  PRECEDENT_NOT_RELIED           106   0.03%  
  PROCEDURAL                   97488  24.74%  ████████████
------------------------------------------------------------
  TOTAL                       394012

STEP 2: ROLE CONTRIBUTION SCORES


...

✅  Annotations saved → ./ensemble_results_real_kg_v4/test_sentence_role_annotations_8label.json (100 docs)

Role                           Count       %  Bar
------------------------------------------------------------
  PREAMBLE                        14   0.10%  
  FACTS                            6   0.04%  
  ISSUE                            3   0.02%  
  ARGUMENTS                     7275  54.26%  ███████████████████████████
  REASONING                     2820  21.03%  ██████████
  PRECEDENT_RELIED                15   0.11%  
  PRECEDENT_NOT_RELIED             7   0.05%  
  PROCEDURAL                    3268  24.37%  ████████████
------------------------------------------------------------
  TOTAL                        13408

STEP 5: GENERATING SUMMARIES — FULL PIPELINE v4

🔄 Processing documents...


Documents:   0%|                                                                                | 0/100 [00:00<?, ?it/s]/home/RSlab/.conda/envs/legal-nlp/lib/python3.10/site-packages/spacy/util.py:969: UserWarning: [W095] Model 'en_legal_ner_trf' (3.2.0) was trained with spaCy v3.2.2 and may not be 100% compatible with the current version (3.8.11). If you see errors or degraded performance, download a newer compatible model or retrain your custom model with the current spaCy version. For more details and available updates, run: python -m spacy validate
  warnings.warn(warn_msg)
/home/RSlab/.conda/envs/legal-nlp/lib/python3.10/site-packages/spacy_transformers/layers/hf_shim.py:137: UserWarning: Error loading saved torch state_dict with strict=True, likely due to differences between 'transformers' versions. Attempting to load with strict=False as a fallback...

If you see errors or degraded performance, download a newer compatible model or retrain your custom model with the current 'tran

✅  All summaries generated!

📊 KG + factual logs saved to: ./ensemble_results_real_kg_v4/

📊 EVALUATING ALL APPROACHES

  Evaluating BART...


/home/RSlab/.conda/envs/legal-nlp/lib/python3.10/site-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  R1:0.3627  R2:0.1819  RL:0.2032  BS:0.8491

  Evaluating LED...
  R1:0.5024  R2:0.2653  RL:0.2754  BS:0.8520

  Evaluating PEGASUS...
  R1:0.3752  R2:0.1880  RL:0.2101  BS:0.8431

  Evaluating LED_kg_cent ★...
  R1:0.5024  R2:0.2653  RL:0.2754  BS:0.8520

  Evaluating Ensemble Voting...
  R1:0.4385  R2:0.2166  RL:0.2215  BS:0.8427

  Evaluating Ensemble Weighted...
  R1:0.4152  R2:0.2068  RL:0.2282  BS:0.8458

  Evaluating Ensemble Best (oracle)...
  R1:0.4993  R2:0.2800  RL:0.2808  BS:0.8569

  Evaluating Ensemble Ranking...
  R1:0.4980  R2:0.2468  RL:0.2416  BS:0.8369

  Evaluating Pseudo-KG Refined (baseline)...
  R1:0.5024  R2:0.2653  RL:0.2754  BS:0.8520

  Evaluating LCS-Fused (Idea 8)...
  R1:0.5031  R2:0.2653  RL:0.2738  BS:0.8515

  Evaluating Verbatim Injected (Idea 9)...
  R1:0.4688  R2:0.2406  RL:0.2567  BS:0.8461

  Evaluating ★ Graph Coverage KG-Diff (Idea 12)...
  R1:0.5024  R2:0.2653  RL:0.2754  BS:0.8520

  Evaluating Structure-Aware Paraphrase (Idea 11)...
  R1:0.46